In [ ]:
!pip install pylate
!pip install --upgrade torchvision


In [ ]:
!pip uninstall -y torch torchvision transformers sentence-transformers pylate
!pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
!pip install pylate


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import gc
import sqlite3
import time

import torch
from pylate import indexes, models
from tqdm.auto import tqdm

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Load the ColBERT model
model = models.ColBERT(
    model_name_or_path="lightonai/colbertv2.0",
    device = device
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/669 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/582 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

1_Dense/model.safetensors:   0%|          | 0.00/393k [00:00<?, ?B/s]

In [ ]:
!pip install pylate[voyager]

In [6]:
OVERRIDE_INDEX = True  # set to False to load an existing index

index = indexes.Voyager(
    index_folder="/content/drive/MyDrive/search_modernColBERT/pylate_index",
    index_name="esci_data_index",
    override=OVERRIDE_INDEX,
)

In [7]:
LIMIT = None # set to None to index all documents

conn = sqlite3.connect("/content/drive/MyDrive/search_modernColBERT/products_dataset.db")
cursor = conn.cursor()

query = "SELECT pid, product_text FROM products"
if LIMIT:
    query += f" LIMIT {LIMIT}"
cursor.execute(query)

In [8]:
ENCODE_BATCH_SIZE = 128    # docs per model.encode() call (GPU memory bound)
INDEX_BATCH_SIZE = 10_000  # docs accumulated before calling add_documents (RAM bound)

def iter_batches(cursor, batch_size):
    while chunk := cursor.fetchmany(batch_size):
        ids, texts = zip(*chunk)
        yield list(ids), list(texts)

def flush_to_index(ids, embeddings):
    index.add_documents(documents_ids=ids, documents_embeddings=embeddings)
    ids.clear()
    embeddings.clear()

total_start = time.time()
encoding_time = 0
indexing_time = 0
pending_ids = []
pending_embeddings = []

n_batches = (LIMIT // ENCODE_BATCH_SIZE) if LIMIT else None
for batch_ids, batch_texts in tqdm(iter_batches(cursor, ENCODE_BATCH_SIZE), total=n_batches, unit="batch", desc="Encoding"):
    t0 = time.time()
    embeddings = model.encode(
        batch_texts,
        is_query=False,
        batch_size=ENCODE_BATCH_SIZE,
        pool_factor=2,
        show_progress_bar=False,
    )
    if isinstance(embeddings, list):
        embeddings = [e.cpu() if hasattr(e, "cpu") else e for e in embeddings]
    encoding_time += time.time() - t0

    pending_ids.extend(batch_ids)
    pending_embeddings.extend(embeddings)

    if len(pending_ids) >= INDEX_BATCH_SIZE:
        print(f"Flushing {len(pending_ids)} documents to index...")
        t0 = time.time()
        flush_to_index(pending_ids, pending_embeddings)
        indexing_time += time.time() - t0

if pending_ids:
    print(f"Flushing final {len(pending_ids)} documents to index...")
    t0 = time.time()
    flush_to_index(pending_ids, pending_embeddings)
    indexing_time += time.time() - t0

conn.close()
gc.collect()

print(f"Encoding time: {encoding_time:.1f}s")
print(f"Indexing time: {indexing_time:.1f}s")
print(f"Total time:    {time.time() - total_start:.1f}s")

Encoding: 0batch [00:00, ?batch/s]

Flushing 10112 documents to index...


Adding documents to the index (bs=2000): 100%|██████████| 6/6 [06:09<00:00, 61.67s/it]


Flushing 10112 documents to index...


Adding documents to the index (bs=2000): 100%|██████████| 6/6 [06:41<00:00, 66.85s/it]


...

Flushing 10112 documents to index...


Adding documents to the index (bs=2000): 100%|██████████| 6/6 [06:12<00:00, 62.07s/it]


Flushing 10112 documents to index...


Adding documents to the index (bs=2000):   0%|          | 0/6 [01:13<?, ?it/s]


KeyboardInterrupt: 